# TCC — Preparação do Dataset de Caracteres Japoneses

Notebook responsável por extrair, filtrar e organizar as imagens dos datasets ETL para o treinamento da CNN do projeto KanjiScan.

Os dados brutos são provenientes do **ETL Character Database** (ETL1, ETL6, ETL8B, ETL9B), um acervo público de caracteres japoneses manuscritos e impressos mantido pelo AIST. Este notebook seleciona apenas os caracteres relevantes ao escopo do projeto, organiza as imagens por pasta de classe e gera metadados para rastreabilidade.

---

In [ ]:
!unzip /content/drive/MyDrive/TCC/output.zip -d /content/

A saída de streaming foi truncada nas últimas 5000 linhas.
  inflating: /content/output/ETL9B/977/143.jpg  
  inflating: /content/output/ETL9B/977/144.jpg  
  inflating: /content/output/ETL9B/977/145.jpg  
  inflating: /content/output/ETL9B/977/146.jpg  
  inflating: /content/output/ETL9B/977/147.jpg  
  inflating: /content/output/ETL9B/977/148.jpg  
  inflating: /content/output/ETL9B/977/149.jpg  
  inflating: /content/output/ETL9B/977/15.jpg  
  inflating: /content/output/ETL9B/977/150.jpg  
  inflating: /content/output/ETL9B/977/151.jpg  
  inflating: /content/output/ETL9B/977/152.jpg  
  inflating: /content/output/ETL9B/977/153.jpg  
  inflating: /content/output/ETL9B/977/154.jpg  
  inflating: /content/output/ETL9B/977/155.jpg  
  inflating: /content/output/ETL9B/977/156.jpg  
  inflating: /content/output/ETL9B/977/157.jpg  
  inflating: /content/output/ETL9B/977/158.jpg  
  inflating: /content/output/ETL9B/977/159.jpg  
  inflating: /content/output/ETL9B/977/16.jpg  
  inflating:

## 1. Definição dos Caracteres Alvo

Define o conjunto de caracteres japoneses que serão incluídos no dataset. O escopo cobre três sistemas de escrita:

| Grupo | Descrição | Exemplos |
|---|---|---|
| Hiragana | Silabário básico + variações sonoras (dakuten/handakuten) | あ、が、ぱ |
| Hiragana pequenos | Formas reduzidas usadas em combinações | ゃ、ゅ、ょ、っ |
| Katakana | Silabário fonético (usado para palavras estrangeiras) | ア、カ、サ |
| Katakana pequenos | Formas reduzidas do katakana | ァ、ィ、ュ、ッ |
| Kanji JLPT N5 | Ideogramas do nível iniciante do exame JLPT | 日、本、語、人 |

`target_chars` é um `set` Python — a verificação `char in target_chars` é O(1), o que é importante no loop de varredura dos datasets.

In [ ]:
hiragana = list("あいうえおかきくけこさしすせそたちつてとなにぬねのはひふへほまみむめもやゆよらりるれろわをん")

hiragana_voiced = list("がぎぐげござじずぜぞだぢづでどばびぶべぼぱぴぷぺぽ")

hiragana_small = list("ゃゅょっ")

katakana = list("アイウエオカキクケコサシスセソタチツテトナニヌネノハヒフヘホマミムメモヤユヨラリルレロワヲン")

katakana_small = list("ァィゥェォャュョッ")

jlpt_n5_kanji = list("一二三四五六七八九十百千万日月火水木金土年時分半今上下中大小右左前後外男女子父母友人先生学校本語文字名何山川田天気電車駅食飲見聞読書話行来帰休買入出会白黒赤青円店国家室週毎午間東西南北")

target_chars = set(
    hiragana +
    hiragana_voiced +
    hiragana_small +
    katakana +
    katakana_small +
    jlpt_n5_kanji
)

## 2. Imports e Configuração dos Diretórios

Carrega as bibliotecas necessárias e define a estrutura de diretórios:

- `BASE_DIR`: raiz dos datasets ETL extraídos (`/content/output`)
- `OUTPUT_DIR`: destino do dataset processado (`/content/dataset_processed`)

A estrutura de saída segue o formato `ImageFolder` do PyTorch, onde cada subdiretório representa uma classe:

```
dataset_processed/
└── images/
    ├── あ/      ← imagens do caractere あ
    ├── い/
    ├── ...
    └── 黒/
```

In [ ]:
import os
import re
import json
import shutil
import pandas as pd
from pathlib import Path
from tqdm import tqdm

BASE_DIR = Path("/content/output")
OUTPUT_DIR = Path("/content/dataset_processed")

DATASETS = ["ETL1", "ETL6", "ETL8B", "ETL9B"]

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "images").mkdir(parents=True, exist_ok=True)

## 3. Parser do Arquivo de Encoding

Cada dataset ETL acompanha um arquivo `encoding.txt` que mapeia caracteres para pastas numéricas e contagens de imagens. O formato do arquivo é:

```
np.str_('あ'): ['1', 1411]
np.str_('い'): ['2', 2822]
```

A função `parse_encoding_file` usa regex para extrair esses mapeamentos e retorna um dicionário `{char: {folder, count}}`.

In [ ]:
def parse_encoding_file(path):

    text = Path(path).read_text(encoding="utf-8")

    pattern = r"np\.str_\('(.+?)'\): \['(\d+)', (\d+)\]"
    matches = re.findall(pattern, text)

    encoding = {}

    for char, folder, count in matches:
        encoding[char] = {
            "folder": folder,
            "count": int(count)
        }

    return encoding

## 4. Construção dos Metadados

Varre os quatro datasets (ETL1, ETL6, ETL8B, ETL9B), lê os arquivos de encoding e filtra apenas os caracteres presentes em `target_chars`. Para cada entrada válida, registra:

- Qual dataset (`etl`) fornece o caractere
- Em qual pasta numérica (`folder`) as imagens estão armazenadas
- Quantas imagens eram esperadas vs. quantas foram efetivamente encontradas no disco

O resultado é consolidado em um DataFrame e salvo em `metadata.csv` para rastreabilidade. Entradas com caminhos inexistentes são descartadas com um aviso.

In [ ]:
rows = []

for dataset_name in DATASETS:
    encoding_path = BASE_DIR / dataset_name / "encoding.txt"

    if not encoding_path.exists():
        print(f"Encoding não encontrado: {encoding_path}")
        continue

    encoding = parse_encoding_file(encoding_path)

    for char, info in encoding.items():
        if char not in target_chars:
            continue

        folder = info["folder"]
        count = info["count"]
        source_path = BASE_DIR / dataset_name / folder

        if not source_path.exists():
            print(f"Pasta não encontrada: {source_path}")
            continue

        image_files = list(source_path.glob("*"))

        rows.append({
            "etl": dataset_name,
            "character": char,
            "folder": folder,
            "expected_count": count,
            "found_count": len(image_files),
            "source_path": str(source_path)
        })

metadata = pd.DataFrame(rows)
metadata.head()
metadata.to_csv(OUTPUT_DIR / "metadata.csv", index=False, encoding="utf-8-sig")
metadata

,etl,character,folder,expected_count,found_count,source_path
0,ETL1,ア,49,1411,1411,/content/output/ETL1/49
1,ETL1,ィ,50,1411,1411,/content/output/ETL1/50
2,ETL1,イ,51,2822,2822,/content/output/ETL1/51
3,ETL1,ウ,52,2822,2822,/content/output/ETL1/52
4,ETL1,ェ,53,1411,1411,/content/output/ETL1/53
...,...,...,...,...,...,...
419,ETL9B,青,2879,201,201,/content/output/ETL9B/2879
420,ETL9B,食,2926,201,201,/content/output/ETL9B/2926
421,ETL9B,飲,2929,201,201,/content/output/ETL9B/2929
422,ETL9B,駅,2949,201,201,/content/output/ETL9B/2949


### Utilitário: Sanitização de Nomes de Arquivo

Função auxiliar que substitui por `_` qualquer caractere não alfanumérico em um nome de arquivo. Garante que nomes gerados a partir de metadados (que podem conter caracteres especiais) sejam seguros para o sistema de arquivos.

In [ ]:
def safe_filename(name):
    return re.sub(r"[^\w\-_.]", "_", name)

## 5. Cópia e Organização das Imagens

Itera sobre os metadados e copia cada imagem para a pasta da sua classe dentro de `OUTPUT_DIR/images/`. O nome de destino segue o padrão `{etl}_{folder}_{idx:05d}{ext}`, garantindo unicidade entre diferentes datasets.

Apenas extensões de imagem válidas são copiadas (`.png`, `.jpg`, `.jpeg`, `.bmp`, `.tif`, `.tiff`). Subdiretórios e outros arquivos são ignorados.

O resultado completo é salvo em `processed_metadata.csv`, mapeando cada arquivo original ao seu destino.

In [ ]:
processed_rows = []

for _, row in tqdm(metadata.iterrows(), total=len(metadata)):
    char = row["character"]
    etl = row["etl"]
    folder = row["folder"]
    source_path = Path(row["source_path"])

    target_char_dir = OUTPUT_DIR / "images" / char
    target_char_dir.mkdir(parents=True, exist_ok=True)

    image_files = sorted(source_path.glob("*"))

    for idx, img_path in enumerate(image_files):
        if img_path.is_dir():
            continue

        ext = img_path.suffix.lower()

        if ext not in [".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"]:
            continue

        new_name = f"{etl}_{folder}_{idx:05d}{ext}"
        target_path = target_char_dir / new_name

        shutil.copy2(img_path, target_path)

        processed_rows.append({
            "character": char,
            "etl": etl,
            "original_path": str(img_path),
            "processed_path": str(target_path)
        })

processed_metadata = pd.DataFrame(processed_rows)
processed_metadata.to_csv(OUTPUT_DIR / "processed_metadata.csv", index=False, encoding="utf-8-sig")

processed_metadata.head()

100%|██████████| 424/424 [01:31<00:00,  4.62it/s]


,character,etl,original_path,processed_path
0,ア,ETL1,/content/output/ETL1/49/0.jpg,/content/dataset_processed/images/ア/ETL1_49_00...
1,ア,ETL1,/content/output/ETL1/49/1.jpg,/content/dataset_processed/images/ア/ETL1_49_00...
2,ア,ETL1,/content/output/ETL1/49/10.jpg,/content/dataset_processed/images/ア/ETL1_49_00...
3,ア,ETL1,/content/output/ETL1/49/100.jpg,/content/dataset_processed/images/ア/ETL1_49_00...
4,ア,ETL1,/content/output/ETL1/49/1000.jpg,/content/dataset_processed/images/ア/ETL1_49_00...


## 6. Análise da Distribuição de Classes

Conta o número de imagens por classe no dataset processado. Uma distribuição equilibrada é desejável para que o modelo não desenvolva viés em favor de classes com mais amostras — um problema conhecido como *class imbalance*.

In [ ]:
class_counts = processed_metadata["character"].value_counts().sort_index()

class_counts

,count
character,
あ,362
い,362
う,362
え,362
お,362
...,...
青,362
食,362
飲,362


### Estatísticas Gerais do Dataset

Sumariza o total de classes, total de imagens e os valores mínimo e máximo de amostras por classe. A diferença entre `min` e `max` indica o grau de desbalanceamento — classes com muito mais amostras tendem a dominar o gradiente durante o treinamento.

In [ ]:
print("Total de classes:", len(class_counts))
print("Total de imagens:", len(processed_metadata))
print("Menor classe:", class_counts.min())
print("Maior classe:", class_counts.max())

Total de classes: 215
Total de imagens: 195227
Menor classe: 161
Maior classe: 4205


## 7. Empacotamento e Exportação para o Google Drive

Comprime o diretório `dataset_processed/` em um arquivo `.zip` e o copia para o Google Drive, tornando-o persistente entre sessões do Colab. O arquivo gerado é utilizado diretamente pelo notebook de treinamento (`TCC_Training.ipynb`).

In [ ]:
!zip -r dataset_processed.zip /content/dataset_processed

A saída de streaming foi truncada nas últimas 5000 linhas.
  adding: content/dataset_processed/images/ヨ/ETL6_103_01061.jpg (stored 0%)
  adding: content/dataset_processed/images/ヨ/ETL6_103_00056.jpg (stored 0%)
  adding: content/dataset_processed/images/ヨ/ETL1_88_00394.jpg (stored 0%)
  adding: content/dataset_processed/images/ヨ/ETL1_88_00825.jpg (stored 0%)
  adding: content/dataset_processed/images/ヨ/ETL1_88_00880.jpg (stored 0%)
  adding: content/dataset_processed/images/ヨ/ETL6_103_00013.jpg (stored 0%)
  adding: content/dataset_processed/images/ヨ/ETL1_88_00335.jpg (stored 0%)
  adding: content/dataset_processed/images/ヨ/ETL6_103_00247.jpg (stored 0%)
  adding: content/dataset_processed/images/ヨ/ETL1_88_00130.jpg (stored 0%)
  adding: content/dataset_processed/images/ヨ/ETL1_88_01184.jpg (stored 0%)
  adding: content/dataset_processed/images/ヨ/ETL6_103_00262.jpg (stored 0%)
  adding: content/dataset_processed/images/ヨ/ETL6_103_00902.jpg (stored 0%)
  adding: content/dataset_processed

In [ ]:
!cp /content/dataset_processed.zip /content/drive/MyDrive/TCC